# Accessing [BioModels](https://www.ebi.ac.uk/biomodels/) with BioServices

**BioModels** is a repository of mathematical models of biological and
biomedical systems.  Models are encoded in standard formats such as SBML.

This notebook shows how to:

- Search and browse available models
- Retrieve model metadata
- Download model files

> **BioModels REST API**: https://www.ebi.ac.uk/biomodels/


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from bioservices import BioModels

s = BioModels()


## 1. Searching for models

Use `search` to find models by keyword.  The result contains model metadata
such as identifier, name and number of reactions/species.


In [ ]:
# Search for models related to MAPK signalling
results = s.search("MAPK")
print(type(results))
if isinstance(results, dict):
    models = results.get("models", [])
    print(f"Found {len(models)} models matching 'MAPK'")
    for m in models[:5]:
        print(f"  {m.get('id')}: {m.get('name')}")
else:
    print(results)


## 2. Retrieving model metadata

Use `get_model` to fetch detailed metadata for a specific model by its
BioModels identifier (e.g. `MODEL1305240000`).


In [ ]:
# Retrieve metadata for a specific model
biomodel = s.get_model("BIOMD0000000001")
print("Model ID   :", biomodel.get("publicationId") or biomodel.get("id", "n/a"))
print("Model name :", biomodel.get("name"))
print("Authors    :", biomodel.get("submitter", "n/a"))
desc = biomodel.get("description", "")
print("Description (first 200 chars):", str(desc)[:200])


## 3. Listing model files

A BioModels entry may include several associated files (SBML, MATLAB scripts,
data files, etc.).


In [ ]:
# List files available for model BIOMD0000000001
files = s.get_model_files("BIOMD0000000001")
if isinstance(files, dict):
    for category, entries in files.items():
        print(f"[{category}]")
        if isinstance(entries, list):
            for e in entries:
                print(f"  {e.get('name', e)}")
        else:
            print(f"  {entries}")
else:
    print(files)


## 4. Downloading a model file

Use `get_model_download` to save the SBML file to disk.


In [ ]:
import tempfile, os

# Download the SBML file to a temporary location
with tempfile.TemporaryDirectory() as tmpdir:
    dest = os.path.join(tmpdir, "BIOMD0000000001.xml")
    s.get_model_download("BIOMD0000000001", "BIOMD0000000001.xml", output_filename=dest)
    size = os.path.getsize(dest)
    print(f"Downloaded SBML file: {dest} ({size} bytes)")
    with open(dest) as fh:
        content = fh.read()
    print("First 400 characters:")
    print(content[:400])


## 5. Browsing the model list

Retrieve a batch of curated models and inspect their properties.


In [ ]:
# Get the first page of curated models
all_models = s.search("*", numResults=20)
if isinstance(all_models, dict):
    model_list = all_models.get("models", [])
    df = pd.DataFrame([
        {"id": m.get("id"), "name": m.get("name"), "format": m.get("format", "SBML")}
        for m in model_list
    ])
    print(f"Loaded {len(df)} models")
    print(df.head(10).to_string(index=False))


---
For more information, please see the
[bioservices.biomodels module documentation](https://bioservices.readthedocs.io/en/main/references.html#module-bioservices.biomodels)
and the [BioModels website](https://www.ebi.ac.uk/biomodels/).
